<a href="https://colab.research.google.com/github/aayushiie/insured-model/blob/main/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
df = pd.read_csv('medical_insurance_dataset.csv')

In [3]:
df.head()

,record_date,year,quarter,age,age_group,sex,sex_female,bmi,bmi_category,children,...,region,region_northeast,region_northwest,region_southeast,region_southwest,charges,monthly_premium_est,charges_per_child,insurance_tier,bmi_age_interaction
0,2024-02-01,2024,1,19,Young Adult (18-25),female,1,27.90,Overweight,0,...,southwest,0,0,0,1,16884.92,1407.08,0.00,Platinum,530.10
1,2024-12-30,2024,4,18,Young Adult (18-25),male,0,33.77,Obese Class I,1,...,southeast,0,0,1,0,1725.55,143.80,1725.55,Bronze,607.86
2,2023-05-11,2023,2,28,Adult (26-35),male,0,33.00,Obese Class I,3,...,southeast,0,0,1,0,4449.46,370.79,1483.15,Silver,924.00
3,2024-07-18,2024,3,33,Adult (26-35),male,0,22.70,Normal Weight,0,...,northwest,0,1,0,0,21984.47,1832.04,0.00,Diamond,749.10
4,2024-02-05,2024,1,32,Adult (26-35),male,0,28.88,Overweight,0,...,northwest,0,1,0,0,3866.86,322.24,0.00,Bronze,924.16


In [4]:
df.shape

(1337, 24)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1337 entries, 0 to 1336
Data columns (total 24 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   record_date          1337 non-null   object 
 1   year                 1337 non-null   int64  
 2   quarter              1337 non-null   int64  
 3   age                  1337 non-null   int64  
 4   age_group            1337 non-null   object 
 5   sex                  1337 non-null   object 
 6   sex_female           1337 non-null   int64  
 7   bmi                  1337 non-null   float64
 8   bmi_category         1337 non-null   object 
 9   children             1337 non-null   int64  
 10  smoker               1337 non-null   object 
 11  smoker_flag          1337 non-null   int64  
 12  is_high_risk         1337 non-null   int64  
 13  risk_score           1337 non-null   float64
 14  region               1337 non-null   object 
 15  region_northeast     1337 non-null   i

In [6]:
df.iloc[1, :]

,1
record_date,2024-12-30
year,2024
quarter,4
age,18
age_group,Young Adult (18-25)
sex,male
sex_female,0
bmi,33.77
bmi_category,Obese Class I
children,1


In [7]:
df = df.drop(columns=['record_date', 'year', 'quarter', 'age', 'sex', 'bmi_category', 'smoker', 'region_northwest', 'region_northeast', 'region_southeast', 'region_southwest'])

In [8]:
df.head()

,age_group,sex_female,bmi,children,smoker_flag,is_high_risk,risk_score,region,charges,monthly_premium_est,charges_per_child,insurance_tier,bmi_age_interaction
0,Young Adult (18-25),1,27.90,0,1,1,5.35,southwest,16884.92,1407.08,0.00,Platinum,530.10
1,Young Adult (18-25),0,33.77,1,0,1,1.40,southeast,1725.55,143.80,1725.55,Bronze,607.86
2,Adult (26-35),0,33.00,3,0,1,2.18,southeast,4449.46,370.79,1483.15,Silver,924.00
3,Adult (26-35),0,22.70,0,0,0,1.11,northwest,21984.47,1832.04,0.00,Diamond,749.10
4,Adult (26-35),0,28.88,0,0,0,1.48,northwest,3866.86,322.24,0.00,Bronze,924.16


In [9]:
df = df.rename(columns={'sex_female': 'sex', 'smoker_flag': 'smoker'})

In [10]:
X = df.drop(columns=['insurance_tier'])

In [11]:
y = df.loc[:, 'insurance_tier']

In [12]:
X.head()

,age_group,sex,bmi,children,smoker,is_high_risk,risk_score,region,charges,monthly_premium_est,charges_per_child,bmi_age_interaction
0,Young Adult (18-25),1,27.90,0,1,1,5.35,southwest,16884.92,1407.08,0.00,530.10
1,Young Adult (18-25),0,33.77,1,0,1,1.40,southeast,1725.55,143.80,1725.55,607.86
2,Adult (26-35),0,33.00,3,0,1,2.18,southeast,4449.46,370.79,1483.15,924.00
3,Adult (26-35),0,22.70,0,0,0,1.11,northwest,21984.47,1832.04,0.00,749.10
4,Adult (26-35),0,28.88,0,0,0,1.48,northwest,3866.86,322.24,0.00,924.16


In [13]:
categorical_features = ['age_group', 'region']
numerical_features = ['bmi', 'risk_score', 'charges', 'monthly_premium_est', 'charges_per_child', 'bmi_age_interaction']

In [14]:
preprocessor = ColumnTransformer(
    transformers = [
        ('cat', OneHotEncoder(), categorical_features),
        ('num', StandardScaler(), numerical_features)
    ]
)

In [15]:
pipeline = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group', 'region']),
                                                 ('num', StandardScaler(),
                                                  ['bmi', 'risk_score',
                                                   'charges',
                                                   'monthly_premium_est',
                                                   'charges_per_child',
                                                   'bmi_age_interaction'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [18]:
y_pred = pipeline.predict(X_test)
accuracy_score(y_test, y_pred)

0.9888059701492538

In [19]:
report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

      Bronze       1.00      1.00      1.00        55
     Diamond       1.00      0.98      0.99        59
        Gold       0.96      1.00      0.98        51
    Platinum       0.98      0.96      0.97        53
      Silver       1.00      1.00      1.00        50

    accuracy                           0.99       268
   macro avg       0.99      0.99      0.99       268
weighted avg       0.99      0.99      0.99       268



In [ ]:
# downloading the model
import pickle
pickle_model_path = "model.pkl"
with open(pickle_model_path, 'wb') as f:
    pickle.dump(pipeline, f)

In [20]:
X['age_group'].unique()

array(['Young Adult (18-25)', 'Adult (26-35)', 'Senior-Middle (46-55)',
       'Middle-Aged (36-45)', 'Senior (56+)'], dtype=object)

In [21]:
X['risk_score'].max()

8.32

In [22]:
X['risk_score'].min()

0.0